# OIBSIP Data Analytics — Task 3: Cleaning Data

This notebook documents data-quality issues, justified treatments, duplicate removal, formatting standardization, IQR outlier detection, dtype correction and a before/after summary.

In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path

df=pd.read_csv('../data/messy_retail_data.csv')
print('Initial shape:',df.shape)
print('Duplicates:',df.duplicated().sum())
quality=pd.DataFrame({'dtype':df.dtypes.astype(str),'nulls':df.isna().sum(),'unique':df.nunique()})
display(quality)

Initial shape: (12080, 21)
Duplicates: 80


,dtype,nulls,unique
Invoice_ID,int64,0,12000
Invoice_Date,object,0,11863
City,object,0,8
Store_Format,object,0,3
Category,object,0,8
Brand,object,0,8
Channel,object,0,3
Payment_Mode,object,0,4
Units,float64,0,10
Cost_Price,float64,0,12000


## Data quality report
Quality checks cover missing values, duplicates, data types and plausible ranges. The messy file intentionally contains inconsistent gender formatting, numeric fields stored as text, invalid revenue strings, negative units and duplicated rows.

In [2]:
before_rows=len(df); before_dups=df.duplicated().sum()
# Standardize categorical formatting
if 'Customer_Gender' in df:
    df['Customer_Gender']=df['Customer_Gender'].astype('string').str.strip().str.upper().replace({'MALE':'M','FEMALE':'F','UNKNOWN':'U','NAN':pd.NA})
# Date parsing
df['Invoice_Date']=pd.to_datetime(df['Invoice_Date'],errors='coerce')
# Numeric coercion
for c in ['Units','Revenue','Cost_Price','Selling_Price','Cost','Margin','Margin_%','Stock_On_Hand','Reorder_Level','Lead_Time_Days','Customer_Age']:
    if c in df: df[c]=pd.to_numeric(df[c],errors='coerce')
# Range anomalies: negative units are invalid for sales transactions -> missing, then impute by median
neg_units=(df['Units']<0).sum(); df.loc[df['Units']<0,'Units']=np.nan
# Missing-value strategy: median for numeric operational fields; mode for categorical gender; invalid dates are dropped because date is required for time-based records
for c in df.select_dtypes(include=np.number).columns:
    df[c]=df[c].fillna(df[c].median())
if 'Customer_Gender' in df: df['Customer_Gender']=df['Customer_Gender'].fillna(df['Customer_Gender'].mode()[0])
df=df.dropna(subset=['Invoice_Date'])
# Duplicate removal
df=df.drop_duplicates().copy()
print('Negative Units found:',neg_units)
print('Duplicates removed:',before_dups)
print('After shape:',df.shape)

Negative Units found: 60
Duplicates removed: 80
After shape: (12000, 21)


## Missing-value strategy justification
- **Numeric sales/operations fields:** median imputation is robust to skew and extreme values common in sales data.
- **Customer_Gender:** mode imputation preserves the categorical type without inventing a new numeric value.
- **Invoice_Date:** rows with unparseable dates are removed because a fabricated date would distort time-series analysis.
- **Negative Units:** treated as a range anomaly for a sales-quantity field and imputed after being converted to missing.

In [3]:
# IQR outlier detection on key numeric fields
outlier_rows=[]
for c in ['Units','Revenue','Margin']:
    q1,q3=df[c].quantile([.25,.75]); iqr=q3-q1; lo=q1-1.5*iqr; hi=q3+1.5*iqr
    n=((df[c]<lo)|(df[c]>hi)).sum(); outlier_rows.append([c,q1,q3,iqr,lo,hi,n])
out=pd.DataFrame(outlier_rows,columns=['Column','Q1','Q3','IQR','Lower_Bound','Upper_Bound','Outlier_Count']); display(out)

,Column,Q1,Q3,IQR,Lower_Bound,Upper_Bound,Outlier_Count
0,Units,2.000000,4.000000,2.000000,-1.000000,7.000000,0
1,Revenue,156.261476,581.647955,425.386479,-481.818243,1219.727674,120
2,Margin,24.372790,111.643339,87.270550,-106.533035,242.549164,584


## Outlier decision
IQR is used to **detect**, not automatically delete, outliers. Legitimate large transactions are possible in retail data, so extreme observations are retained unless they are demonstrably erroneous. This avoids deleting valid high-value sales.

In [4]:
# Correct dtypes and final quality summary
for c in ['Units','Stock_On_Hand','Reorder_Level','Lead_Time_Days']: df[c]=df[c].astype(int)
for c in ['Revenue','Cost_Price','Selling_Price','Cost','Margin','Margin_%','Customer_Age']: df[c]=df[c].astype(float)
after=pd.DataFrame({'metric':['rows','columns','duplicates','null_cells'],'before':[before_rows,quality.shape[0],before_dups,quality.nulls.sum()],'after':[len(df),df.shape[1],df.duplicated().sum(),df.isna().sum().sum()]})
display(after)
df.to_csv('../outputs/cleaned_retail_data.csv',index=False)
after.to_csv('../outputs/before_after_summary.csv',index=False)

,metric,before,after
0,rows,12080,12000
1,columns,21,21
2,duplicates,80,0
3,null_cells,5362,0


## Final conclusion
The cleaned dataset has standardized categorical values, corrected numeric/date types, handled missing values using explicit rules, removed exact duplicates and documented IQR outliers without indiscriminate deletion. The cleaned CSV is saved for downstream analysis.